# Step 7 (rebuilt again) — Three-Lever Search, Service Frozen

D-079. A full-grid diagnostic review of the previous (D-071) two-lever search found `service_target` and `inventory_cover_weeks` are not independently identifiable — both terminate in a single `target_stock` scalar that every cost formula reads only as a total. **`service_target` is removed as a decision lever and reported as an achieved output instead.** It stays on `LeverSettings` (deleting it breaks the Step 6 suite) but is frozen at each SKU's ABC-class default.

The review also found `bias_correction` (always 0.0) and `min_run_hours` (held at a per-line category default, never varied) were the two levers actually left unexplored in the retracted search — confirmed on real data to move cost, with `min_run_hours` dominating by roughly two orders of magnitude.

**New free levers: cover (per class) × bias correction (per line) × min run hours (per line).** Grid: 3×3×3×3×4 = 324 combinations per line — smaller and faster than the retracted 729-combination search it replaces.

## Setup

In [ ]:
import subprocess, os, sys
def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r
REPO = '/content/ibp-tradeoff'
os.chdir('/content'); sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## Rebuild the Step 4 artefacts

In [ ]:
import pandas as pd, numpy as np, yaml
from src.ingest import DataIngestor
from src.cleaner import DataCleaner
if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')
ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

## Upload Step 5a's output

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv:')
uploaded = files.upload()
demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'demand_characteristics: {demand_characteristics.shape} | flagged: {len(flagged)}')

## Build the engine

In [ ]:
from src.engine import TradeOffEngine, LeverSettings, build_line_master
from src.policy_model import (search_all_lines_three_lever, search_three_lever_policy,
                              best_feasible_three_lever, bias_min_run_interaction)
assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)
print('assumption fingerprint:', engine.assumption_fingerprint)
print('frozen service (ABC defaults):', {c: assumptions['abc'][c]['service_floor'] for c in ('A','B','C')})

## Change 1 — confirm bias_correction and min_run_hours actually move cost

Before trusting the new search, the same confirmatory sweep run in sandbox: hold each line's cover at a sensible point, vary bias × min_run, confirm both move total cost within a single line.

In [ ]:
BIAS_GRID = [0.0, 0.5, 1.0]
MRH_GRID = [4.0, 6.0, 9.0, 12.0]
rows = []
for line_id in line_master['line_id']:
    skus = engine.line_skus(line_id)
    if not skus: continue
    category = str(engine.sku_master.loc[skus[0], 'category'])
    default_cover = {c: float(assumptions['abc'][c]['target_cover_weeks']) for c in ('A','B','C')}
    frozen_service = {c: float(assumptions['abc'][c]['service_floor']) for c in ('A','B','C')}
    for bias in BIAS_GRID:
        for mrh in MRH_GRID:
            levers = LeverSettings(service_target=frozen_service, inventory_cover_weeks=default_cover,
                                   forecast_bias_correction=bias, min_run_hours=mrh).validate(assumptions)
            res = engine.run_scenario(line_id, levers)
            rows.append({'line_id':line_id,'bias':bias,'min_run_hours':mrh,
                        'total_economic_cost_eur':res.totals['total_economic_cost_eur']})
change1 = pd.DataFrame(rows)
for line_id in line_master['line_id']:
    sub = change1[change1.line_id==line_id]
    if len(sub)==0: continue
    print(f'{line_id}: range {sub.total_economic_cost_eur.min():,.0f} - {sub.total_economic_cost_eur.max():,.0f}  (spread {sub.total_economic_cost_eur.max()-sub.total_economic_cost_eur.min():,.0f})')

## The new search — cover x bias x min_run, service frozen

In [ ]:
import time
t0 = time.time()
line_results = search_all_lines_three_lever(engine)
print(f'search_all_lines_three_lever took {time.time()-t0:.0f}s')
line_results.to_csv('line_results_three_lever.csv', index=False)

cols = ['line_id','cover_A','cover_B','cover_C','bias_correction','min_run_hours',
        'total_economic_cost_eur','default_total_cost_eur','saving_eur',
        'service_achieved_A','service_achieved_B','service_achieved_C','unit_fill_rate',
        'overhang_cost_share','capacity_shortfall_total']
print(line_results[cols].round(4).to_string(index=False))
print(f"\ntotal saving vs true base: EUR {line_results.saving_eur.sum():,.0f}")
print(f"minimum saving (must be >= 0): {line_results.saving_eur.min():,.2f}")
print(f"any infeasible optimum (must be 0): {(line_results.capacity_shortfall_total > 1e-6).sum()}")

## Change 3 — achieved service by class, and unit fill rate

Already computed per row above (`service_achieved_A/B/C`, `unit_fill_rate` — the aggregate `1 - sum(lost)/sum(demand)`, never a mean of per-row ratios). Shown here for the winning row of each line.

In [ ]:
print(line_results[['line_id','service_achieved_A','service_achieved_B','service_achieved_C','unit_fill_rate']].round(4).to_string(index=False))

## Change 4 — overhang diagnostic

Share of SKU-months where `production_units == 0` and `stock_open_units > target_stock_units`, and the share of cost arising in those months. Reporting only — does not alter the inventory balance (D-038 stands).

In [ ]:
print(line_results[['line_id','overhang_cost_share']].round(4).to_string(index=False))

## Bias x min-run interaction, including null cells

At each line's own winning cover combination. Cells where bias has exactly zero effect are flagged, not suppressed.

In [ ]:
interactions = {}
for line_id in line_master['line_id']:
    row = line_results[line_results.line_id==line_id]
    if len(row)==0: continue
    r = row.iloc[0]
    cover_combo = {'A': r.cover_A, 'B': r.cover_B, 'C': r.cover_C}
    inter = bias_min_run_interaction(engine, line_id, cover_combo)
    interactions[line_id] = inter
    print(f'--- {line_id} ---')
    print(inter.round(0).to_string(index=False))
    print()

## Change 5 — delta vs base, fixed absorption stripped

In [ ]:
from src.reporter import delta_vs_base_view
FIXED_CONV = float(assumptions['plant_economics']['fixed_absorption_eur_line_month']) * 12
deltas = []
for _, row in line_results.iterrows():
    d = delta_vs_base_view(row.to_dict(), base_total_eur=row.default_total_cost_eur, fixed_conversion_eur=FIXED_CONV)
    deltas.append(d)
delta_df = pd.DataFrame(deltas)
print(delta_df.round(2).to_string(index=False))
delta_df.to_csv('delta_vs_base.csv', index=False)

## Tests

In [ ]:
sh('python -m pytest tests/test_policy_model.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_reporter.py tests/test_pipeline.py -q --no-header')

## Save outputs to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
out_dir = '/content/drive/My Drive/ibp-tradeoff-outputs'
os.makedirs(out_dir, exist_ok=True)
for f in ['line_results_three_lever.csv', 'delta_vs_base.csv']:
    if os.path.exists(f): shutil.copy(f, out_dir)
print('copied to', out_dir)

## Consolidated report — the only cell to copy

In [ ]:
import hashlib, subprocess

t_pol = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                       shell=True, capture_output=True, text=True)
t_eng = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                       shell=True, capture_output=True, text=True)
t_rep = subprocess.run('python -m pytest tests/test_reporter.py -q --no-header',
                       shell=True, capture_output=True, text=True)
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 7 (D-079) - THREE-LEVER SEARCH, SERVICE FROZEN - CONSOLIDATED REPORT'); w('='*78)
w(f'assumption set    : {engine.assumption_fingerprint}')
w(f'policy_model sha  : {hashlib.sha256(open("src/policy_model.py","rb").read()).hexdigest()[:12]}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')

w(''); w('-- 1. CHANGE 1 CONFIRMATION - bias/min_run move cost within a line '+'-'*10)
for line_id in line_master['line_id']:
    sub = change1[change1.line_id==line_id]
    if len(sub): w(f'{line_id}: spread {sub.total_economic_cost_eur.max()-sub.total_economic_cost_eur.min():,.0f}')

w(''); w('-- 2. LINE RESULTS - three-lever feasible optimum '+'-'*30)
w(line_results[cols].round(2).to_string(index=False))
w(f'\\ntotal saving vs true base: EUR {line_results.saving_eur.sum():,.0f}')

w(''); w('-- 3. FEASIBILITY '+'-'*60)
w(f'minimum saving_eur (must be >= 0): {line_results.saving_eur.min():,.2f}')
w(f'rows with capacity_shortfall_total > 0 (must be 0): {(line_results.capacity_shortfall_total > 1e-6).sum()}')

w(''); w('-- 4. ACHIEVED SERVICE BY CLASS (Change 3) '+'-'*35)
w(line_results[['line_id','service_achieved_A','service_achieved_B','service_achieved_C','unit_fill_rate']].round(4).to_string(index=False))

w(''); w('-- 5. OVERHANG DIAGNOSTIC (Change 4) '+'-'*40)
w(line_results[['line_id','overhang_cost_share']].round(4).to_string(index=False))

w(''); w('-- 6. BIAS x MIN-RUN INTERACTION, null cells included '+'-'*22)
for line_id, inter in interactions.items():
    w(f'{line_id}:')
    w(inter.round(0).to_string(index=False))

w(''); w('-- 7. DELTA VS BASE, fixed absorption stripped (Change 5) '+'-'*18)
w(delta_df.round(2).to_string(index=False))

w(''); w('-- 8. TESTS '+'-'*66)
for label, r in (('test_policy_model.py', t_pol), ('test_engine.py', t_eng),
                 ('test_reporter.py', t_rep), ('test_pipeline.py', t_pipe)):
    w(f'{label:<24}: ' + (r.stdout.strip().splitlines() or ["no output"])[-1])
if any(r.returncode for r in (t_pol, t_eng, t_rep, t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_pol, t_eng, t_rep, t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 9. CHECKS '+'-'*65)
checks = [
 ('every optimum is capacity-feasible', bool((line_results.capacity_shortfall_total <= 1e-6).all())),
 ('optimum never worse than true base', bool(line_results.saving_eur.min() >= -1e-6)),
 ('service is frozen, not searched (no service columns in grid)', 'service_A' not in line_results.columns),
 ('bias x min-run null cells present and shown', any(inter['bias_has_zero_effect'].any() for inter in interactions.values())),
 ('all test suites pass', all(r.returncode == 0 for r in (t_pol, t_eng, t_rep, t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step07_d079_report.txt','w').write(report_text)
try:
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step07_d079_report.txt','line_results_three_lever.csv','delta_vs_base.csv'):
            if os.path.exists(f): shutil.copy(f, d)
        print('saved to', d, '\\n')
except Exception as e:
    print('Drive copy skipped:', e, '\\n')
print(report_text)